In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_excel('Dry_Bean_Dataset.xlsx')
print('Shape:', df.shape)
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after duplicate removal:', df.shape)

In [ ]:
print(df['Class'].value_counts())

plt.figure(figsize=(10,5))
sns.countplot(data=df, x='Class')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df.hist(figsize=(18,15), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(df.drop('Class', axis=1).corr(), cmap='coolwarm')
plt.show()

In [ ]:
for col in df.columns[:-1]:
    plt.figure(figsize=(8,3))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

In [ ]:
le = LabelEncoder()
df['Class'] = le.fit_transform(df['Class'])

X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10,6))
scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=y)
plt.colorbar(scatter)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.show()

In [ ]:
scores = []

for k in range(1,31):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores.append(accuracy_score(y_test, pred))

plt.figure(figsize=(10,5))
plt.plot(range(1,31), scores, marker='o')
plt.grid()
plt.show()

best_k = np.argmax(scores) + 1
print('Best K:', best_k)

In [ ]:
param_grid = {
    'n_neighbors':[3,5,7,9,11,13,15],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan','minkowski']
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('Best Parameters:', grid.best_params_)
print('Best CV Score:', grid.best_score_)

In [ ]:
final_knn = grid.best_estimator_

final_knn.fit(X_train, y_train)

final_pred = final_knn.predict(X_test)

print('Final Accuracy:', accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

In [ ]:
cm = confusion_matrix(y_test, final_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.show()

In [ ]:
joblib.dump(final_knn, 'drybean_knn_classifier.pkl')
print('Model Saved')

In [ ]:
sample = X_test[0].reshape(1,-1)
prediction = final_knn.predict(sample)

print('Predicted Class:', le.inverse_transform(prediction)[0])